# Flight Delay Prediction with Transformer Architecture

This notebook implements a transformer-based model for predicting flight delays using the historical flights dataset. The architecture is adapted from a ship position prediction model and modified for flight delay forecasting.

## Key Features:
- **Multi-modal inputs**: Flight sequences + categorical embeddings (airports, airlines)
- **Circular encodings**: Time-based features encoded as sin/cos pairs
- **Transformer attention**: Captures temporal dependencies in flight operations
- **Multi-task learning**: Predicts both delay duration and cancellation probability

## Architecture Overview:
1. **Sequence Generation**: Create flight sequences per aircraft/route
2. **Feature Engineering**: Circular encodings for temporal features
3. **Transformer Encoder**: Self-attention mechanism for sequence modeling
4. **Categorical Embeddings**: Airport and airline representations
5. **Multi-head Output**: Regression (delay minutes) + Classification (cancellation)

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import os

warnings.filterwarnings("ignore")

# TensorFlow configuration
print(f"TensorFlow version: {tf.__version__}")
print(f"Available CPUs: {os.cpu_count()}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

2025-09-15 16:23:55.100754: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-15 16:23:55.101061: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-15 16:23:55.140108: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-15 16:23:56.035232: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-poin

TensorFlow version: 2.20.0
Available CPUs: 32
GPU available: []


2025-09-15 16:23:56.718813: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
# Model Configuration
SEQUENCE_LENGTH = 10  # Number of historical flights per sequence
NUM_TARGETS = 2  # [delay_minutes, is_cancelled]
FUTURE_HORIZON = 1  # Predict next flight delay

# Transformer parameters
D_MODEL = 64  # Model dimension
NUM_HEADS = 8  # Number of attention heads
FF_DIM = 256  # Feed-forward dimension
NUM_TRANSFORMER_BLOCKS = 4
DROPOUT_RATE = 0.1

# Training parameters
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

print(f"Configuration:")
print(f"- Sequence Length: {SEQUENCE_LENGTH}")
print(f"- Model Dimension: {D_MODEL}")
print(f"- Attention Heads: {NUM_HEADS}")
print(f"- Transformer Blocks: {NUM_TRANSFORMER_BLOCKS}")

Configuration:
- Sequence Length: 10
- Model Dimension: 64
- Attention Heads: 8
- Transformer Blocks: 4


In [3]:
# Load Historical Flights Dataset
print("Loading historical flights dataset...")
data_path = "/lhome/kristsey/AvinorHackathon/data/historical_flights.csv"

# Load data in chunks due to large size
chunk_size = 50000
chunks = []
total_rows = 0

for chunk in pd.read_csv(data_path, chunksize=chunk_size):
    chunks.append(chunk)
    total_rows += len(chunk)
    if total_rows >= 100000:  # Limit for testing
        break
    print(f"Loaded {total_rows:,} rows...")

# Combine chunks
df = pd.concat(chunks, ignore_index=True)
print(f"Final dataset shape: {df.shape}")

# Display basic information
print("\nDataset columns:")
print(df.columns.tolist())
print(f"\nFirst few rows:")
print(df.head())

Loading historical flights dataset...
Loaded 50,000 rows...
Final dataset shape: (100000, 11)

Dataset columns:
['flight_id', 'dep_airport', 'dep_airport_group', 'arr_airport', 'arr_airport_group', 'service_type', 'std', 'sta', 'cancelled', 'atd', 'ata']

First few rows:
  flight_id dep_airport dep_airport_group arr_airport arr_airport_group  \
0     WF149         HOV                 B         OSL               NaN   
1     WF722         OSL               NaN         MJF                 D   
2     WF188         FDE                 A         OSL               NaN   
3     WF176         HOV                 B         OSL               NaN   
4     WF148         HOV                 B         OSL               NaN   

  service_type                         std                         sta  \
0            J  2018-01-02T16:40:00.000000  2018-01-02T17:15:00.000000   
1            J  2018-01-28T13:04:00.000000  2018-01-28T14:50:00.000000   
2            J  2018-04-07T07:10:00.000000  2018-04-07T

In [4]:
# Feature Engineering and Preprocessing
print("Starting feature engineering...")

# Create a copy for processing
df_processed = df.copy()

# Convert datetime columns
datetime_cols = ["std", "sta", "atd", "ata"]
for col in datetime_cols:
    if col in df_processed.columns:
        df_processed[col] = pd.to_datetime(df_processed[col], errors="coerce")


# Calculate delay metrics
def calculate_delay_minutes(row):
    if pd.isna(row["atd"]) or pd.isna(row["std"]):
        return None
    delay_minutes = (row["atd"] - row["std"]).total_seconds() / 60
    return max(-60, min(600, delay_minutes))  # Clip between -60 and 600 minutes


def calculate_is_cancelled(row):
    return 1 if row["cancelled"] == 1 else 0


# Create target variables
df_processed["delay_minutes"] = df_processed.apply(calculate_delay_minutes, axis=1)
df_processed["is_cancelled"] = df_processed.apply(calculate_is_cancelled, axis=1)

# Remove rows without delay information
df_processed = df_processed.dropna(subset=["delay_minutes"])
print(f"Dataset shape after removing missing delays: {df_processed.shape}")

# Extract temporal features
df_processed["dep_hour"] = df_processed["std"].dt.hour
df_processed["dep_day_of_week"] = df_processed["std"].dt.dayofweek
df_processed["dep_month"] = df_processed["std"].dt.month
df_processed["dep_day_of_year"] = df_processed["std"].dt.dayofyear

# Calculate scheduled duration
df_processed["scheduled_duration"] = (
    df_processed["sta"] - df_processed["std"]
).dt.total_seconds() / 60

# Circular encodings for temporal features
print("Creating circular encodings...")

# Hour (0-23)
df_processed["dep_hour_sin"] = np.sin(2 * np.pi * df_processed["dep_hour"] / 24)
df_processed["dep_hour_cos"] = np.cos(2 * np.pi * df_processed["dep_hour"] / 24)

# Day of week (0-6)
df_processed["dep_day_of_week_sin"] = np.sin(
    2 * np.pi * df_processed["dep_day_of_week"] / 7
)
df_processed["dep_day_of_week_cos"] = np.cos(
    2 * np.pi * df_processed["dep_day_of_week"] / 7
)

# Month (1-12)
df_processed["dep_month_sin"] = np.sin(2 * np.pi * (df_processed["dep_month"] - 1) / 12)
df_processed["dep_month_cos"] = np.cos(2 * np.pi * (df_processed["dep_month"] - 1) / 12)

# Day of year (1-366)
df_processed["dep_day_of_year_sin"] = np.sin(
    2 * np.pi * (df_processed["dep_day_of_year"] - 1) / 366
)
df_processed["dep_day_of_year_cos"] = np.cos(
    2 * np.pi * (df_processed["dep_day_of_year"] - 1) / 366
)

# Create time difference for temporal sequences (hours between flights)
df_processed = df_processed.sort_values(["flight_id", "std"])
df_processed["time_diff"] = (
    df_processed.groupby("dep_airport")["std"].diff().dt.total_seconds() / 3600
)
df_processed["time_diff"] = df_processed["time_diff"].fillna(0)

print("Feature engineering completed!")
print(f"Target statistics:")
print(f"- Average delay: {df_processed['delay_minutes'].mean():.2f} minutes")
print(f"- Cancellation rate: {df_processed['is_cancelled'].mean():.2%}")
print(f"- Total samples: {len(df_processed):,}")

Starting feature engineering...
Dataset shape after removing missing delays: (96729, 13)
Creating circular encodings...
Feature engineering completed!
Target statistics:
- Average delay: 3.20 minutes
- Cancellation rate: 0.02%
- Total samples: 96,729
Dataset shape after removing missing delays: (96729, 13)
Creating circular encodings...
Feature engineering completed!
Target statistics:
- Average delay: 3.20 minutes
- Cancellation rate: 0.02%
- Total samples: 96,729


In [5]:
# Create Flight Sequences for Transformer Input
print("Creating flight sequences...")


def create_flight_sequences(data, sequence_length=SEQUENCE_LENGTH):
    """
    Create sequences of flights grouped by route (dep_airport -> arr_airport)
    Similar to vessel sequences in ship prediction
    """
    # Select features for sequences
    feature_columns = [
        "dep_hour_sin",
        "dep_hour_cos",
        "dep_day_of_week_sin",
        "dep_day_of_week_cos",
        "dep_month_sin",
        "dep_month_cos",
        "dep_day_of_year_sin",
        "dep_day_of_year_cos",
        "scheduled_duration",
        "time_diff",
    ]

    # Normalize numerical features
    scaler = MinMaxScaler()
    data_scaled = data.copy()
    data_scaled[feature_columns] = scaler.fit_transform(data_scaled[feature_columns])

    # Group by route
    data_scaled["route"] = data_scaled["dep_airport"] + "_" + data_scaled["arr_airport"]

    sequences_X = []
    sequences_Y = []
    routes = []
    dep_airports = []
    arr_airports = []

    # Create sequences for each route
    route_groups = data_scaled.groupby("route")

    for route, group in route_groups:
        if len(group) < sequence_length + 1:
            continue

        # Sort by time
        group = group.sort_values("std")

        # Create overlapping sequences
        for i in range(len(group) - sequence_length):
            # Input sequence (features)
            sequence_features = group.iloc[i : i + sequence_length][
                feature_columns
            ].values

            # Target (next flight's delay and cancellation)
            next_flight = group.iloc[i + sequence_length]
            targets = [next_flight["delay_minutes"], next_flight["is_cancelled"]]

            sequences_X.append(sequence_features)
            sequences_Y.append(targets)
            routes.append(route)
            dep_airports.append(next_flight["dep_airport"])
            arr_airports.append(next_flight["arr_airport"])

    print(f"Created {len(sequences_X)} sequences from {len(route_groups)} routes")
    return (
        np.array(sequences_X),
        np.array(sequences_Y),
        np.array(routes),
        np.array(dep_airports),
        np.array(arr_airports),
        scaler,
    )


# Create sequences
sequences_X, sequences_Y, routes, dep_airports, arr_airports, feature_scaler = (
    create_flight_sequences(df_processed)
)

# Prepare label encoders for categorical features
route_encoder = LabelEncoder()
dep_airport_encoder = LabelEncoder()
arr_airport_encoder = LabelEncoder()

# Fit encoders
unique_routes = np.unique(routes)
unique_dep_airports = np.unique(dep_airports)
unique_arr_airports = np.unique(arr_airports)

route_encoder.fit(unique_routes)
dep_airport_encoder.fit(unique_dep_airports)
arr_airport_encoder.fit(unique_arr_airports)

# Transform categorical features
routes_encoded = route_encoder.transform(routes)
dep_airports_encoded = dep_airport_encoder.transform(dep_airports)
arr_airports_encoded = arr_airport_encoder.transform(arr_airports)

print(f"Sequence data shape: {sequences_X.shape}")
print(f"Target data shape: {sequences_Y.shape}")
print(f"Number of unique routes: {len(unique_routes)}")
print(f"Number of unique departure airports: {len(unique_dep_airports)}")
print(f"Number of unique arrival airports: {len(unique_arr_airports)}")

# Split into train/validation/test sets
(
    X_train,
    X_temp,
    y_train,
    y_temp,
    routes_train,
    routes_temp,
    dep_train,
    dep_temp,
    arr_train,
    arr_temp,
) = train_test_split(
    sequences_X,
    sequences_Y,
    routes_encoded,
    dep_airports_encoded,
    arr_airports_encoded,
    test_size=0.3,
    random_state=42,
)

(
    X_val,
    X_test,
    y_val,
    y_test,
    routes_val,
    routes_test,
    dep_val,
    dep_test,
    arr_val,
    arr_test,
) = train_test_split(
    X_temp, y_temp, routes_temp, dep_temp, arr_temp, test_size=0.5, random_state=42
)

print(f"\nDataset splits:")
print(f"- Training: {X_train.shape[0]} sequences")
print(f"- Validation: {X_val.shape[0]} sequences")
print(f"- Test: {X_test.shape[0]} sequences")

Creating flight sequences...
Created 95006 sequences from 288 routes
Sequence data shape: (95006, 10, 10)
Target data shape: (95006, 2)
Number of unique routes: 137
Number of unique departure airports: 39
Number of unique arrival airports: 40

Dataset splits:
- Training: 66504 sequences
- Validation: 14251 sequences
- Test: 14251 sequences
Created 95006 sequences from 288 routes
Sequence data shape: (95006, 10, 10)
Target data shape: (95006, 2)
Number of unique routes: 137
Number of unique departure airports: 39
Number of unique arrival airports: 40

Dataset splits:
- Training: 66504 sequences
- Validation: 14251 sequences
- Test: 14251 sequences


In [6]:
# Positional Encoding Layer (adapted from ship prediction model)
@tf.keras.utils.register_keras_serializable()
class FlightPositionalEncoding(layers.Layer):
    def __init__(self, d_model, **kwargs):
        """
        Positional encoding for flight sequences based on time differences.
        Adapted from ContinuousPositionalEncoding in ship prediction model.
        """
        super(FlightPositionalEncoding, self).__init__(**kwargs)
        self.d_model = d_model

    @tf.function
    def call(self, time_differences):
        """
        Computes positional encoding based on time differences between flights.

        Args:
            time_differences: Tensor of shape (batch_size, sequence_length)
                            containing time gaps between consecutive flights

        Returns:
            Positional encoding tensor of shape (batch_size, sequence_length, d_model)
        """
        # Compute cumulative time positions
        time_positions = tf.cumsum(time_differences, axis=1) * SEQUENCE_LENGTH
        position = tf.expand_dims(time_positions, axis=-1)

        # Compute angle rates
        i = tf.range(self.d_model, dtype=tf.float32)
        i = tf.reshape(i, (1, 1, self.d_model))
        angle_rates = 1 / tf.pow(
            10000.0, (2 * (i // 2)) / tf.cast(self.d_model, tf.float32)
        )
        angle_rads = position * angle_rates

        # Apply sin to even indices and cos to odd indices
        sines = tf.sin(angle_rads[..., 0::2])
        cosines = tf.cos(angle_rads[..., 1::2])

        # Concatenate sines and cosines
        pos_encoding = tf.concat([sines, cosines], axis=-1)

        return pos_encoding

    def get_config(self):
        config = super().get_config()
        config.update({"d_model": self.d_model})
        return config

In [7]:
# Transformer Encoder Block
def transformer_encoder_block(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    """
    Single transformer encoder block with multi-head attention and feed-forward layers.
    """
    # Multi-head self-attention
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=head_size, dropout=dropout
    )(inputs, inputs)

    # Skip connection and layer norm
    attention_output = layers.Dropout(dropout)(attention_output)
    attention_output = layers.LayerNormalization(epsilon=1e-6)(
        inputs + attention_output
    )

    # Feed-forward network
    ffn_output = layers.Dense(ff_dim, activation="relu")(attention_output)
    ffn_output = layers.Dropout(dropout)(ffn_output)
    ffn_output = layers.Dense(inputs.shape[-1])(ffn_output)

    # Skip connection and layer norm
    ffn_output = layers.Dropout(dropout)(ffn_output)
    return layers.LayerNormalization(epsilon=1e-6)(attention_output + ffn_output)

In [8]:
# Build Flight Delay Transformer Model
def build_flight_transformer_model(
    sequence_length,
    num_features,
    num_targets,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    num_transformer_blocks=NUM_TRANSFORMER_BLOCKS,
    num_routes=None,
    num_dep_airports=None,
    num_arr_airports=None,
    route_embedding_dim=32,
    airport_embedding_dim=16,
    dropout=DROPOUT_RATE,
):
    """
    Build transformer model for flight delay prediction.
    Similar structure to ship position prediction but adapted for flights.
    """
    # Input layers
    sequence_input = layers.Input(
        shape=(sequence_length, num_features), name="sequence_features"
    )
    route_input = layers.Input(shape=(), dtype=tf.int32, name="route_id")
    dep_airport_input = layers.Input(shape=(), dtype=tf.int32, name="dep_airport_id")
    arr_airport_input = layers.Input(shape=(), dtype=tf.int32, name="arr_airport_id")

    # Extract time differences for positional encoding
    time_diff_index = -1  # time_diff is the last feature
    time_differences = sequence_input[:, :, time_diff_index]

    # Project sequence features to model dimension
    projected_features = layers.Dense(d_model)(sequence_input)

    # Add positional encoding based on time differences
    pos_encoding = FlightPositionalEncoding(d_model)(time_differences)
    sequence_with_pos = projected_features + pos_encoding

    # Apply dropout
    sequence_with_pos = layers.Dropout(dropout)(sequence_with_pos)

    # Stack transformer encoder blocks
    encoded_sequence = sequence_with_pos
    for _ in range(num_transformer_blocks):
        encoded_sequence = transformer_encoder_block(
            encoded_sequence,
            head_size=d_model // num_heads,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout=dropout,
        )

    # Global average pooling to get sequence representation
    sequence_repr = layers.GlobalAveragePooling1D()(encoded_sequence)

    # Categorical embeddings
    route_embedding = layers.Embedding(num_routes, route_embedding_dim)(route_input)
    route_embedding = layers.Flatten()(route_embedding)

    dep_airport_embedding = layers.Embedding(num_dep_airports, airport_embedding_dim)(
        dep_airport_input
    )
    dep_airport_embedding = layers.Flatten()(dep_airport_embedding)

    arr_airport_embedding = layers.Embedding(num_arr_airports, airport_embedding_dim)(
        arr_airport_input
    )
    arr_airport_embedding = layers.Flatten()(arr_airport_embedding)

    # Concatenate all representations
    concatenated = layers.Concatenate()(
        [sequence_repr, route_embedding, dep_airport_embedding, arr_airport_embedding]
    )

    # Dense layers for final prediction
    dense = layers.Dense(256, activation="relu")(concatenated)
    dense = layers.Dropout(dropout)(dense)
    dense = layers.Dense(128, activation="relu")(dense)
    dense = layers.Dropout(dropout)(dense)

    # Multi-task outputs
    delay_output = layers.Dense(1, activation="linear", name="delay_minutes")(dense)
    cancellation_output = layers.Dense(1, activation="sigmoid", name="is_cancelled")(
        dense
    )

    # Combine outputs
    outputs = layers.Concatenate(name="combined_output")(
        [delay_output, cancellation_output]
    )

    # Define model
    model = keras.Model(
        inputs=[sequence_input, route_input, dep_airport_input, arr_airport_input],
        outputs=outputs,
        name="FlightDelayTransformer",
    )

    return model


# Build the model
print("Building transformer model...")

model = build_flight_transformer_model(
    sequence_length=SEQUENCE_LENGTH,
    num_features=X_train.shape[2],  # Number of features per timestep
    num_targets=NUM_TARGETS,
    num_routes=len(unique_routes),
    num_dep_airports=len(unique_dep_airports),
    num_arr_airports=len(unique_arr_airports),
)

# Compile model with multi-task loss
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse",  # Since we have 2 targets: delay_minutes (regression) + is_cancelled (classification)
    metrics=["mae"],
)

# Display model summary
model.summary()

Building transformer model...


Model: "FlightDelayTransformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_features   │ (None, 10, 10)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 10)        │          0 │ sequence_feature… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10, 64)    │        704 │ sequence_feature… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flight_positional_… │ (None, 10, 64)    │          0 │ get_item[0][0]    │
│ (FlightPositionalE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 64)    │          0 │ dense[0][0],      │
│                     │                   │            │ flight_positiona… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 10, 64)    │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 64)    │     16,640 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 10, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 10, 64)    │          0 │ dropout[0][0],    │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 10, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10, 256)   │     16,640 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 10, 256)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 10, 64)    │     16,448 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 10, 64)    │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 10, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 64)    │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 10, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 10, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 64)    │        128 │ add_3[0][0]     

 Total params: 272,466 (1.04 MB)

 Trainable params: 272,466 (1.04 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Prepare Training Data and Train Model
print("Preparing training datasets...")


def create_tf_dataset(
    X, y, routes, dep_airports, arr_airports, batch_size, shuffle=True
):
    """Create TensorFlow dataset for training."""
    dataset = tf.data.Dataset.from_tensor_slices(
        ((X, routes, dep_airports, arr_airports), y)
    )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=10000)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


# Create datasets
train_dataset = create_tf_dataset(
    X_train, y_train, routes_train, dep_train, arr_train, BATCH_SIZE
)
val_dataset = create_tf_dataset(
    X_val, y_val, routes_val, dep_val, arr_val, BATCH_SIZE, shuffle=False
)

# Callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.2, patience=5, verbose=1, min_lr=1e-7
)

# Train model
print("Starting training...")
print(f"Training on {X_train.shape[0]} sequences for {EPOCHS} epochs...")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping, reduce_lr],
    verbose=1,
)

print("Training completed!")

Preparing training datasets...
Starting training...
Training on 66504 sequences for 50 epochs...
Epoch 1/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 36s 14ms/step - loss: 155.7753 - mae: 5.1351 - val_loss: 166.5125 - val_mae: 5.1611 - learning_rate: 0.0010
Epoch 2/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 36s 14ms/step - loss: 155.7753 - mae: 5.1351 - val_loss: 166.5125 - val_mae: 5.1611 - learning_rate: 0.0010
Epoch 2/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 30s 14ms/step - loss: 154.4207 - mae: 5.0957 - val_loss: 166.4762 - val_mae: 5.2466 - learning_rate: 0.0010
Epoch 3/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 30s 14ms/step - loss: 154.4207 - mae: 5.0957 - val_loss: 166.4762 - val_mae: 5.2466 - learning_rate: 0.0010
Epoch 3/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 30s 14ms/step - loss: 154.3656 - mae: 5.0959 - val_loss: 166.4916 - val_mae: 5.2365 - learning_rate: 0.0010
Epoch 4/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 30s 14ms/step - loss: 154.3656 - mae: 5.0959 - val_loss: 166.4916 - val_mae: 5.2365 - learning_rate: 0.0010
Epo

In [ ]:
# Model Evaluation and Results
print("Evaluating model performance...")

# Make predictions on test set
test_dataset = create_tf_dataset(
    X_test, y_test, routes_test, dep_test, arr_test, BATCH_SIZE, shuffle=False
)
predictions = model.predict(test_dataset)

# Split predictions (first column: delay_minutes, second column: is_cancelled)
pred_delay = predictions[:, 0]
pred_cancellation = predictions[:, 1]

# Split actual targets
actual_delay = y_test[:, 0]
actual_cancellation = y_test[:, 1]

# Regression metrics (delay prediction)
delay_mae = mean_absolute_error(actual_delay, pred_delay)
delay_rmse = np.sqrt(np.mean((actual_delay - pred_delay) ** 2))

# Classification metrics (cancellation prediction)
pred_cancellation_binary = (pred_cancellation > 0.5).astype(int)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

cancellation_accuracy = accuracy_score(actual_cancellation, pred_cancellation_binary)
cancellation_precision = precision_score(actual_cancellation, pred_cancellation_binary)
cancellation_recall = recall_score(actual_cancellation, pred_cancellation_binary)
cancellation_f1 = f1_score(actual_cancellation, pred_cancellation_binary)

# Print results
print("=== Model Performance ===")
print(f"\nDelay Prediction (Regression):")
print(f"  - Mean Absolute Error: {delay_mae:.2f} minutes")
print(f"  - Root Mean Square Error: {delay_rmse:.2f} minutes")
print(
    f"  - Actual delay range: {actual_delay.min():.1f} to {actual_delay.max():.1f} minutes"
)

print(f"\nCancellation Prediction (Classification):")
print(f"  - Accuracy: {cancellation_accuracy:.4f}")
print(f"  - Precision: {cancellation_precision:.4f}")
print(f"  - Recall: {cancellation_recall:.4f}")
print(f"  - F1-Score: {cancellation_f1:.4f}")
print(f"  - Actual cancellation rate: {actual_cancellation.mean():.2%}")

# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Training history
axes[0, 0].plot(history.history["loss"], label="Training Loss")
axes[0, 0].plot(history.history["val_loss"], label="Validation Loss")
axes[0, 0].set_title("Training History")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True)

# 2. Delay prediction scatter plot
axes[0, 1].scatter(actual_delay, pred_delay, alpha=0.5)
axes[0, 1].plot(
    [actual_delay.min(), actual_delay.max()],
    [actual_delay.min(), actual_delay.max()],
    "r--",
)
axes[0, 1].set_title("Delay Prediction: Actual vs Predicted")
axes[0, 1].set_xlabel("Actual Delay (minutes)")
axes[0, 1].set_ylabel("Predicted Delay (minutes)")
axes[0, 1].grid(True)

# 3. Delay prediction residuals
residuals = actual_delay - pred_delay
axes[1, 0].hist(residuals, bins=50, alpha=0.7)
axes[1, 0].set_title("Delay Prediction Residuals")
axes[1, 0].set_xlabel("Residual (Actual - Predicted)")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].grid(True)

# 4. Cancellation prediction distribution
axes[1, 1].hist(
    pred_cancellation[actual_cancellation == 0],
    bins=30,
    alpha=0.5,
    label="Not Cancelled",
    color="green",
)
axes[1, 1].hist(
    pred_cancellation[actual_cancellation == 1],
    bins=30,
    alpha=0.5,
    label="Cancelled",
    color="red",
)
axes[1, 1].set_title("Cancellation Probability Distribution")
axes[1, 1].set_xlabel("Predicted Probability")
axes[1, 1].set_ylabel("Frequency")
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

# Compare with simple baseline
baseline_delay_mae = mean_absolute_error(
    actual_delay, np.full_like(actual_delay, actual_delay.mean())
)
baseline_cancellation_accuracy = max(
    actual_cancellation.mean(), 1 - actual_cancellation.mean()
)

print(f"\n=== Baseline Comparison ===")
print(
    f"Delay MAE - Transformer: {delay_mae:.2f} vs Baseline (mean): {baseline_delay_mae:.2f}"
)
print(
    f"Cancellation Accuracy - Transformer: {cancellation_accuracy:.4f} vs Baseline (majority): {baseline_cancellation_accuracy:.4f}"
)

improvement_delay = (baseline_delay_mae - delay_mae) / baseline_delay_mae * 100
improvement_cancellation = (
    (cancellation_accuracy - baseline_cancellation_accuracy)
    / baseline_cancellation_accuracy
    * 100
)

print(f"\nImprovement over baseline:")
print(f"  - Delay prediction: {improvement_delay:.1f}% better")
print(f"  - Cancellation prediction: {improvement_cancellation:.1f}% better")

In [ ]:
# Attention Visualization and Model Analysis
print("Analyzing transformer attention patterns...")


# Extract attention weights from the model
def get_attention_weights(model, sample_input):
    """Extract attention weights from transformer layers."""
    # Create a model that outputs attention weights
    attention_outputs = []

    for layer in model.layers:
        if isinstance(layer, layers.MultiHeadAttention):
            # Get attention scores
            attention_layer = keras.Model(inputs=model.input, outputs=layer.output)
            attention_outputs.append(attention_layer)

    return attention_outputs


# Example prediction with attention analysis
print("\n=== Example Predictions ===")

# Select a few test samples for detailed analysis
sample_indices = [0, 10, 20, 50, 100]
for i in sample_indices:
    if i >= len(X_test):
        continue

    # Single sample prediction
    sample_X = X_test[i : i + 1]
    sample_route = routes_test[i : i + 1]
    sample_dep = dep_test[i : i + 1]
    sample_arr = arr_test[i : i + 1]

    prediction = model.predict(
        [sample_X, sample_route, sample_dep, sample_arr], verbose=0
    )
    actual = y_test[i]

    print(f"\nSample {i+1}:")
    print(f"  Route: {route_encoder.inverse_transform([sample_route[0]])[0]}")
    print(
        f"  Departure Airport: {dep_airport_encoder.inverse_transform([sample_dep[0]])[0]}"
    )
    print(
        f"  Arrival Airport: {arr_airport_encoder.inverse_transform([sample_arr[0]])[0]}"
    )
    print(
        f"  Predicted Delay: {prediction[0][0]:.1f} minutes (Actual: {actual[0]:.1f})"
    )
    print(f"  Predicted Cancellation: {prediction[0][1]:.3f} (Actual: {actual[1]:.0f})")

    # Show sequence characteristics
    sequence_features = sample_X[0]
    avg_scheduled_duration = sequence_features[:, -2].mean()  # scheduled_duration
    avg_time_diff = sequence_features[:, -1].mean()  # time_diff
    print(f"  Avg Scheduled Duration: {avg_scheduled_duration:.1f} minutes")
    print(f"  Avg Time Between Flights: {avg_time_diff:.1f} hours")

# Feature importance analysis using permutation
print("\n=== Feature Importance Analysis ===")

feature_names = [
    "dep_hour_sin",
    "dep_hour_cos",
    "dep_day_of_week_sin",
    "dep_day_of_week_cos",
    "dep_month_sin",
    "dep_month_cos",
    "dep_day_of_year_sin",
    "dep_day_of_year_cos",
    "scheduled_duration",
    "time_diff",
]


def permutation_importance(
    model, X, y, routes, dep_airports, arr_airports, feature_idx, n_samples=1000
):
    """Calculate permutation importance for a specific feature."""
    # Take a subset for faster computation
    indices = np.random.choice(len(X), min(n_samples, len(X)), replace=False)
    X_subset = X[indices]
    y_subset = y[indices]
    routes_subset = routes[indices]
    dep_subset = dep_airports[indices]
    arr_subset = arr_airports[indices]

    # Baseline performance
    baseline_pred = model.predict(
        [X_subset, routes_subset, dep_subset, arr_subset], verbose=0
    )
    baseline_mae = mean_absolute_error(y_subset[:, 0], baseline_pred[:, 0])  # Delay MAE

    # Permute feature and measure performance drop
    X_permuted = X_subset.copy()
    np.random.shuffle(X_permuted[:, :, feature_idx])

    permuted_pred = model.predict(
        [X_permuted, routes_subset, dep_subset, arr_subset], verbose=0
    )
    permuted_mae = mean_absolute_error(y_subset[:, 0], permuted_pred[:, 0])

    importance = permuted_mae - baseline_mae
    return importance


# Calculate importance for each feature (on a subset for speed)
print("Calculating feature importance (this may take a few minutes)...")
importances = []

for i, feature_name in enumerate(feature_names):
    importance = permutation_importance(
        model, X_test, y_test, routes_test, dep_test, arr_test, i
    )
    importances.append(importance)
    print(f"  {feature_name}: {importance:.3f}")

# Plot feature importance
plt.figure(figsize=(10, 6))
sorted_idx = np.argsort(importances)[::-1]
plt.barh(range(len(importances)), [importances[i] for i in sorted_idx])
plt.yticks(range(len(importances)), [feature_names[i] for i in sorted_idx])
plt.xlabel("Permutation Importance (MAE increase)")
plt.title("Feature Importance in Flight Delay Transformer")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n=== Summary ===")
print(f"✅ Transformer model successfully trained on flight sequences")
print(f"✅ Multi-task learning: delay prediction + cancellation prediction")
print(f"✅ Temporal attention: captures sequential patterns in flight operations")
print(f"✅ Categorical embeddings: learns route and airport representations")
print(f"✅ Circular encodings: preserves temporal cyclical patterns")
print(f"✅ Performance: {delay_mae:.1f} min MAE, {cancellation_f1:.3f} F1-score")